In [ ]:
"""exponentially weighted window(well，直接把这个东西理解为指数分布就好，思考你学过的数理统计的知识）

One must specify precisely one of span, center of mass, half-life and alpha to the EW functions:

Span corresponds to what is commonly called an “N-day EW moving average”.
Center of mass has a more physical interpretation and can be thought of in terms of span:
𝑐 =(𝑠 −1)/2
.
Half-life is the period of time for the exponential weight to reduce to one half.
Alpha specifies the smoothing factor directly.

关键公式：σₜ² = λ·σₜ₋₁² + (1-λ)·rₜ₋₁²
σₜ²：t 时刻条件方差
λ：衰减因子（RiskMetrics 推荐日度 0.94，月度 0.97）
rₜ₋₁：t-1 时刻收益率
权重规律：k 天前数据权重 = (1-λ)・λᵏ，呈指数衰减

EWM（指数加权窗口，给近期数据更多权重，给远期数据更少权重，这背后的经济学直觉也很make sense
因此EWM才是实际研究中常用的，只关注最近行情，远古数据对结果的影响微乎其微
span 是最最常用的参数，
df.ewm(span = 20),等价于20日指数加权，span越大越平滑，span越小越灵敏

常用于计算更灵敏的波动率，（better than rolling）

span = (2 / (1 - λ)) - 1
span = (1+λ)/(1-λ)
λ = 0.94 → span ≈ 32.33
λ = 0.97（月频）→ span ≈ 65.67
"""

"""
# 【任务 9】计算 20 日指数加权移动平均（EWMA）并与简单移动平均对比
# 要求：span=20，画出对比图
# --- 你的代码 ---

# ewma = df_returns.ewm(span=20, adjust=False).mean()
# sma = df_returns.rolling(20).mean()
# 
# fig, ax = plt.subplots(figsize=(12, 6))
# ax.plot(ewma['AAPL'], label='EWMA (span=20)', alpha=0.7)
# ax.plot(sma['AAPL'], label='SMA (window=20)', alpha=0.7)
# ax.legend()
# ax.set_title('AAPL: EWMA vs Simple Moving Average')

# 【任务 10】计算 EWMA 波动率（RiskMetrics 模型）
# λ = 0.94, 对应 span = (2/(1-0.94)) - 1 = 32.33
# 要求：
# 1. 计算平方收益率
# 2. 对平方收益率应用 ewm(span=32.33, adjust=False).mean()
# 3. 开方得到波动率，再年化
# --- 你的代码 ---

# lambda_param = 0.94
# span = (2 / (1 - lambda_param)) - 1
# 
# squared_returns = df_returns ** 2
# ewma_var = squared_returns.ewm(span=span, adjust=False).mean()
# ewma_vol = np.sqrt(ewma_var) * np.sqrt(252)
# 
# ewma_vol.plot(title='EWMA Volatility (λ=0.94, Annualized)')

# 【任务 11】对比 EWMA 波动率 vs 简单滚动波动率
# 要求：同一张图画出两者，观察 EWMA 是否对新信息反应更快
# --- 你的代码 ---

# rolling_vol = df_returns.rolling(20).std(ddof=0) * np.sqrt(252)
# 
# fig, ax = plt.subplots(figsize=(12, 6))
# ax.plot(ewma_vol['AAPL'], label='EWMA Vol (λ=0.94)', alpha=0.7)
# ax.plot(rolling_vol['AAPL'], label='Rolling Vol (20d)', alpha=0.7)
# ax.legend()
# ax.set_title('AAPL: EWMA vs Rolling Volatility')

# 【任务 12】计算指数加权协方差（用于 Beta 计算）
# 股票与"市场组合"的协方差 / 市场方差
# 假设 GOOGL 是"市场"（实际应用中应该用指数）
# --- 你的代码 ---
beta_aapl = (
    ewm_cov.xs('AAPL', level=1)['GOOGL'] / 
    ewm_cov.xs('GOOGL', level=1)['GOOGL']
)

print("AAPL 对 GOOGL 的滚动 Beta（最后5天）:")
print(beta_aapl.tail())

# # 计算指数加权协方差矩阵
# ewm_cov = df_returns.ewm(span=32.33, adjust=False).cov(pairwise=True)
🤔这里是pairwise = True 是对所有资产的协方差进行两两计算
# 
# # 提取 AAPL 对 GOOGL 的 Beta
# # 这题较难，涉及 MultiIndex 操作
# # 提示：ewm_cov 是多层索引，需要 xs 提取
beta_aapl = (
    ewm_cov.xs('AAPL', level=1)['GOOGL'] / 
    ewm_cov.xs('GOOGL', level=1)['GOOGL']
)
这个是矩阵式写法，适合用于批量处理数据，而且这样计算的是所有相对的协方差
print("AAPL 对 GOOGL 的滚动 Beta（最后5天）:")
print(beta_aapl.tail())

"""







In [ ]:
"""One key quantitative result is that the new methodology applied
to a risk horizon of three months is more accurate than the exponential moving average scheme 
at a risk horizon of one day. 

and what is the RM 1994 methodology and historical methodology? 
historical methodology：用一年的历史数据来进行梦托卡罗模拟，来计算尾部的概率，其优势为简单和通俗易懂
指数移动平均（ewma exponentially weighted moving average） 为什么更好，
ewma：的重点是一个半衰期参数，日频使用0.94，月频使用0.97

well ， what is the ARCH methodology？
"""

"""# 在 RiskMetrics 类中添加

def ewma_volatility(self, lambda_param: float = 0.94, annualize: bool = True) -> pd.DataFrame:
    """
    计算 EWMA 波动率（RiskMetrics 模型）
    lambda_param: 衰减因子，日频通常 0.94，月频 0.97
    """
    span = (2 / (1 - lambda_param)) - 1
    
    # EWMA 方差
    squared_returns = self.returns ** 2
    ewma_var = squared_returns.ewm(span=span, adjust=False).mean()
    
    # 波动率
    vol = np.sqrt(ewma_var)
    
    if annualize:
        vol = vol * np.sqrt(252)
    
    return vol

def ewma_covariance(self, lambda_param: float = 0.94) -> pd.DataFrame:
    """
    计算 EWMA 协方差矩阵（MultiIndex）
    返回：DataFrame with MultiIndex (date, asset)
    """
    span = (2 / (1 - lambda_param)) - 1
    return self.returns.ewm(span=span, adjust=False).cov()

def rolling_beta(self, asset: str, benchmark: str, window: int = 60) -> pd.Series:
    """
    计算滚动 Beta（资产相对于基准的敏感度）
    Beta = Cov(asset, benchmark) / Var(benchmark)
    """
    # 滚动协方差
    cov = self.returns[asset].rolling(window).cov(self.returns[benchmark])
    # 滚动方差
    var = self.returns[benchmark].rolling(window).var(ddof=0)
    
    return cov / var

def ewma_beta(self, asset: str, benchmark: str, lambda_param: float = 0.94) -> pd.Series:
    """
    计算 EWMA Beta
    """
    span = (2 / (1 - lambda_param)) - 1
    
    # EWMA 协方差
    cov = self.returns[asset].ewm(span=span, adjust=False).cov(self.returns[benchmark])
    # EWMA 方差
    var = self.returns[benchmark].ewm(span=span, adjust=False).var()
    
    return cov / var"""

In [2]:
import pandas as pd
import numpy as np
import os

np.random.seed(42)

dates = pd.date_range('2025-01-01',periods = 252, freq = 'B')
tickers = ['AMZN','AAPL','MSFT','NDVA']
prices = {}

for ticker in tickers:
    returns = np.random.randn(252)*0.02
    prices[ticker] = 100*np.exp(np.cumsum(returns))

df_prices = pd.DataFrame(prices,index = dates)
df_returns = df_prices.pct_change().dropna()

df_prices
    

,AMZN,AAPL,MSFT,NDVA
2025-01-01,100.998379,104.335671,98.707145,100.991519
2025-01-02,100.719476,106.512528,97.750159,101.365548
2025-01-03,102.032660,103.324572,96.598861,99.640241
2025-01-06,105.188455,102.328736,94.943989,101.045641
2025-01-07,104.697002,104.954693,95.036171,99.888998
...,...,...,...,...
2025-12-12,94.599004,109.128391,65.561305,201.368103
2025-12-15,97.998878,111.168675,64.294121,204.199826
2025-12-16,98.795856,115.496125,65.663356,205.813159
2025-12-17,96.335606,112.310305,64.428305,209.531194
